# Prototipo del agente de indexación (Agente A)

Recorrido guiado del prototipo sobre el lote de ejemplo del curso dictado `412711`.

1. Ver los dos grafos.
2. Correr el agente con un único llamado.
3. Leer el reporte.
4. Seguir un contenido nodo por nodo.
5. Inspeccionar los objetos que generó.
6. Medir el resultado contra las respuestas esperadas.

Por defecto corre en modo **simulado**: sin red ni credenciales. La metadata simulada sirve para revisar el flujo, no para juzgar su calidad. La última sección explica cómo usar Amazon Bedrock.

In [ ]:
from pathlib import Path
import json

from IPython.display import Markdown, display

from indexador import indexar
from indexador.config import cargar_config
from indexador.esquemas.entrada import cargar_entrada
from indexador.grafo import GRAFO_LOTE, SUBGRAFO_CONTENIDO

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
LOTE = RAIZ / "ejemplos" / "curso-412711"
SALIDA = RAIZ / "salida"
LOTE

## 1. Los grafos

El grafo del lote agrupa los archivos por contenido y lanza un subgrafo por cada contenido único. Las flechas punteadas son decisiones.

In [ ]:
display(Markdown("```mermaid\n" + GRAFO_LOTE.get_graph().draw_mermaid() + "\n```"))

In [ ]:
display(Markdown("```mermaid\n" + SUBGRAFO_CONTENIDO.get_graph().draw_mermaid() + "\n```"))

## 2. Un único llamado

`indexar(entrada, salida)` lee la carpeta del lote, corre el grafo y escribe `salida/<runId>/`. Cada corrida crea su propia carpeta.

In [ ]:
resultado = indexar(LOTE, SALIDA)
print(resultado.runId)
print(resultado.carpeta)
resultado.reporte.totales

In [ ]:
sorted(str(p.relative_to(resultado.carpeta)) for p in resultado.carpeta.rglob("*") if p.is_file())

## 3. El reporte

El mismo contenido de `reporte.md`: qué entró, qué salió, los ilegibles con su motivo y la razón de cada respaldo propuesto.

In [ ]:
display(Markdown((resultado.carpeta / "reporte.md").read_text(encoding="utf-8")))

## 4. Un contenido nodo por nodo

`stream` muestra qué devuelve cada nodo del subgrafo. Aquí, el contenido del reto 2.

In [ ]:
entrada = cargar_entrada(LOTE)
config = cargar_config()
archivo = next(a for a in entrada.lote.archivos if a.sourceIdentifier == "4542274")
estado_inicial = {
    "contenido": {"contentHash": archivo.core.contentHash, "archivos": [archivo]},
    "entrada": entrada,
    "config": config,
    "ahora": "2026-01-01T00:00:00+00:00",
}

for paso in SUBGRAFO_CONTENIDO.stream(estado_inicial, stream_mode="updates"):
    for nodo, cambios in paso.items():
        claves = sorted(k for k in cambios if k != "pasos")
        print(f"{nodo:22} → {', '.join(claves) or '(sin cambios)'}")

In [ ]:
final = SUBGRAFO_CONTENIDO.invoke(estado_inicial)
final["metadata"]

In [ ]:
for juicio in final["juicios"]:
    print(juicio.codigoRA, juicio.aporta, juicio.caracter, juicio.confianza, "-", juicio.razon)

## 5. Los objetos generados

Los objetos tienen la misma forma que los del mod `learning-catalog`. Un elemento con sus usos, sus respaldos y su proveniencia:

In [ ]:
def leer(nombre):
    return json.loads((resultado.carpeta / "objetos" / nombre).read_text(encoding="utf-8"))

elementos = leer("learning_elements.json")
usos = leer("usages.json")
respaldos = leer("backings.json")
proveniencia = leer("provenance.json")

programa = next(e for e in elementos if e["descriptiveTitle"].startswith("Programa"))
print(json.dumps(programa, ensure_ascii=False, indent=2))
print("Usos:", [(u["weekOrUnit"], u["order"]) for u in usos if u["learningElementId"] == programa["id"]])
print("Respaldos:", [(b["learningOutcomeId"], b["character"], b["modelConfidence"]) for b in respaldos if b["learningElementId"] == programa["id"]])
print("Proveniencia:", sorted({(p["fieldName"], p["origin"]) for p in proveniencia if p["entityId"] == programa["id"]}))

## 6. Medir contra las respuestas esperadas

`esperado.json` dice qué debería producir el agente con cada archivo. La evaluación compara la corrida y calcula el acuerdo por campo. En modo simulado el acuerdo es bajo a propósito: las heurísticas no razonan como un modelo real.

In [ ]:
from indexador.evaluacion import evaluar, evaluacion_markdown

evaluacion = evaluar(resultado.carpeta, LOTE / "esperado.json")
display(Markdown(evaluacion_markdown(evaluacion)))

## Usar Amazon Bedrock

1. Instalar el extra: `pip install -e ".[bedrock]"`.
2. Tener credenciales de AWS con acceso a Bedrock (por ejemplo `aws sso login`).
3. Copiar `indexador/config.yaml`, cambiar `llm.modo` a `"bedrock"` y poner en `llm.modelo` el id de un modelo que soporte tool use.
4. Correr con esa configuración:

```python
resultado = indexar(LOTE, SALIDA, "mi-config.yaml")
```

Si el modelo falla tras los reintentos, el contenido sale como ilegible con el motivo y la corrida sigue.